In [2]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
import seaborn as sns
import pickle as pkl
from tqdm import tqdm
import csv
from joblib import Parallel, delayed
from collections import Counter
from itertools import product
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2, f_regression

In [3]:
X_byte_bigram_all_df = pd.read_csv("byte_files_bigram_df.csv")

X_byte_bigram_all_df.head(2)


,ID,00 00,00 01,00 02,00 03,00 04,00 05,00 06,00 07,00 08,...,?? f7,?? f8,?? f9,?? fa,?? fb,?? fc,?? fd,?? fe,?? ff,?? ??
0,01azqd4InC7m9JpocGv5,274425,1269,1029,1469,1227,1144,1437,1263,1174,...,0,0,0,0,0,0,0,0,0,1819
1,01IsoiSMh5gxyDYTl4CB,21075,752,73,48,175,12,10,11,42,...,0,0,0,0,0,0,0,0,0,8580


In [39]:
with open('featurization/class_labels.pkl', 'rb') as file:
    class_labels=pkl.load(file)

y = class_labels
y = y.rename(columns={"Id": "ID", "Class": "Class"})
final_df = pd.merge(X_byte_bigram_all_df, y, on="ID", how="inner")

X = final_df.drop(["ID", "Class"], axis=1)
y = final_df["Class"]

In [66]:
from sklearn.feature_selection import SelectKBest, chi2, f_regression

# X_features = X_byte_bigram_all_df.drop("ID", axis=1)

select_kbest_object = SelectKBest(score_func=chi2, k=2000)
most_imp_features_byte_bigram = select_kbest_object.fit(X, y)

# 建立分數 DataFrame
most_imp_byte_bigram_feature_score_df = pd.DataFrame(most_imp_features_byte_bigram.scores_)

# 關鍵修正：從 X_features (不含 ID) 提取欄位名稱
most_imp_byte_bigram_columns_df = pd.DataFrame(X.columns)

In [67]:
# 現在兩邊都是 66049 筆，對齊完全正確
byte_bigram_df_important_feature_score = pd.concat([most_imp_byte_bigram_columns_df, most_imp_byte_bigram_feature_score_df], axis=1)
byte_bigram_df_important_feature_score.columns = ["Byte Bigram Top 2000 Feature Names","Byte Bigram Top 2000 Feature Score"]

# 取得真正的 Top 2000
byte_bigram_df_important_feature_score = byte_bigram_df_important_feature_score.nlargest(2000, "Byte Bigram Top 2000 Feature Score")
byte_bigram_df_important_feature_score

,Byte Bigram Top 2000 Feature Names,Byte Bigram Top 2000 Feature Score
66048,?? ??,1.008849e+10
52632,cc cc,3.254310e+08
0,00 00,1.731621e+08
65790,ff ff,2.790252e+07
255,00 ff,9.768750e+06
...,...,...
14668,39 13,7.903158e+05
1003,03 e8,7.901612e+05
65536,ff 01,7.901540e+05
4249,10 89,7.900037e+05


In [68]:
# Getting the list of first 2000 feature names
top_2000_most_imp_byte_bigram_feature_names = list(byte_bigram_df_important_feature_score["Byte Bigram Top 2000 Feature Names"])

# top_2000_byte_bigram_features = dd.concat([X_byte_bigram_all_df["ID"], X_byte_bigram_all_df[top_2000_most_imp_byte_bigram]], axis=1)
top_2000_byte_bigram_features = pd.concat([X_byte_bigram_all_df["ID"], X_byte_bigram_all_df[top_2000_most_imp_byte_bigram_feature_names]], axis=1)

top_2000_byte_bigram_features.to_csv("featurization/featurization_final/top_2000_imp_byte_bigram_df.csv",index=False)

print(top_2000_byte_bigram_features.shape)
top_2000_byte_bigram_features.head(2)

(10868, 2001)


,ID,?? ??,cc cc,00 00,ff ff,00 ff,ff 00,02 02,10 10,f0 f0,...,33 c7,75 01,05 32,13 24,14 20,39 13,03 e8,ff 01,10 89,13 0c
0,01azqd4InC7m9JpocGv5,1819,99,274425,876,1906,1502,74,98,84,...,9,6,5,7,13,3,12,6,8,7
1,01IsoiSMh5gxyDYTl4CB,8580,732,21075,4861,1036,2202,23,195,21,...,0,3,1,1,1,0,6,128,24,43
